# fig-vip-connectome

Reproducibility notebook for **Figure 1 (fig-vip-connectome)**.

Panel C uses the **single Phase-6-audited quantitative dataset** (Pfeffer et al. 2013, n = 11, V1 L2/3 paired recordings). All other panels encode qualitative cross-study consensus categories and deliberately avoid unaudited numerical values.

Source clusters: cluster_05_synaptic_connectivity, cluster_13_neuromodulation. Style: `scripts/shared_style.py`.

In [ ]:
# fig-vip-connectome — VIP synaptic connectivity at a glance
# Self-contained reproducibility notebook.
# Panel C uses ONLY the audited Pfeffer 2013 dataset (n=11).

import sys, os
HERE = os.path.dirname(os.path.abspath("__file__"))
sys.path.insert(0, os.path.normpath(os.path.join(HERE, "..", "..", "scripts")))
from shared_style import COLORS, apply_style, save_figure

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Ellipse
import matplotlib.patches as mpatches

apply_style()

# ---------- DATA ----------
# Qualitative consensus categories (0=absent, 1=weak, 2=moderate, 3=strong)
inputs  = ["Local Pyr","Long-range\ncortical","Thalamic\n(POm/MD/AT)",
           "Basal-forebrain\nACh","Raphe\n5-HT","LC\nNA"]
targets = ["VIP","SST","PV"]
mat = np.array([[2,2,2],[3,1,1],[3,1,2],[3,0,0],[3,0,0],[2,0,0]])

outputs  = ["SST","PV","Pyr (sup.)","Pyr (deep)","NDNF / L1","OLM (CA1)","CCK BC (CA1)"]
contexts = ["Cortex L2/3","Cortex L5/6","Hippocampus CA1"]
out_mat = np.array([
    [3,2,1,np.nan,1,np.nan,np.nan],
    [2,2,1,2,1,np.nan,np.nan],
    [np.nan]*5 + [3,2],
])

# Pfeffer 2013 — audited Phase-6 dataset
ipsq_means = [4.6, 0.6]; ipsq_err = [1.5, 0.2]   # pC
inc_means  = [0.42, 0.06]; inc_err  = [0.14, 0.02]   # pC

# ---------- FIGURE ----------
fig = plt.figure(figsize=(13.5, 10))
gs = fig.add_gridspec(2, 2, hspace=0.45, wspace=0.32)

axA = fig.add_subplot(gs[0,0])
axA.imshow(mat, cmap=plt.cm.Purples, aspect="auto", vmin=0, vmax=3)
axA.set_xticks(range(len(targets)), targets)
axA.set_yticks(range(len(inputs)), inputs)
labels = {0:"–",1:"weak",2:"mod.",3:"strong"}
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        v = mat[i,j]; c = "white" if v>=2 else "black"
        axA.text(j,i,labels[v],ha="center",va="center",color=c)
axA.set_title("A  Afferent drive", loc="left", fontweight="bold")

axB = fig.add_subplot(gs[0,1])
m = np.ma.masked_invalid(out_mat)
axB.imshow(m, cmap=plt.cm.Purples, aspect="auto", vmin=0, vmax=3)
axB.set_xticks(range(len(outputs)), outputs, rotation=35, ha="right")
axB.set_yticks(range(len(contexts)), contexts)
for i in range(out_mat.shape[0]):
    for j in range(out_mat.shape[1]):
        v = out_mat[i,j]
        if np.isnan(v):
            axB.text(j,i,"n.t.",ha="center",va="center",color="#888")
        else:
            c = "white" if v>=2 else "black"
            axB.text(j,i,labels[int(v)],ha="center",va="center",color=c)
axB.set_title("B  VIP outputs", loc="left", fontweight="bold")

axC = fig.add_subplot(gs[1,0])
x = np.arange(2); w = 0.35
axC.bar(x-w/2, ipsq_means, w, yerr=ipsq_err, capsize=4,
        color=[COLORS["SST"], COLORS["Pyr"]], edgecolor="black", label="IPSQ (pC)")
axC.bar(x+w/2, [v*10 for v in inc_means], w, yerr=[e*10 for e in inc_err],
        capsize=4, color=[COLORS["SST"], COLORS["Pyr"]],
        edgecolor="black", hatch="//", label="INC × 10 (pC)")
axC.set_xticks(x, ["VIP→SST","VIP→Pyr"])
axC.set_ylabel("IPSC charge (pC)")
axC.set_title("C  Pfeffer 2013 (audited)", loc="left", fontweight="bold")
axC.legend(frameon=False)

axD = fig.add_subplot(gs[1,1])
axD.set_xlim(0,10); axD.set_ylim(0,10); axD.axis("off")
axD.set_title("D  Cholinergic recruitment (schematic)", loc="left", fontweight="bold")
axD.add_patch(FancyBboxPatch((0.4,7.5),2.4,1.6, boxstyle="round,pad=0.1",
                              fc="#e8e8f4", ec="#444"))
axD.text(1.6,8.3,"Basal forebrain ACh", ha="center", va="center")
axD.add_patch(Ellipse((5,5),2.4,1.6, fc=COLORS["VIP"], ec="black", alpha=0.85))
axD.text(5,5,"VIP / 5-HT3AR", ha="center", va="center", color="white")

fig.suptitle("Fig. 1 (Section 6) — VIP connectivity at a glance",
             y=1.005, fontweight="bold")

save_figure(fig, "../fig_sec6_connectome.png")
plt.show()
